# Loopback Capture Example

This notebook shows how to use `Experiment.capture_loopback(...)` to capture full-span loopback waveforms from READ_IN and MNTR_IN channels.

`capture_loopback` temporarily switches RF ports to loopback mode, executes the schedule, and restores the original RF-switch states automatically.

In [ ]:
import numpy as np

import qubex as qx

In [ ]:
# Initialize experiment session (replace paths/chip/qubits for your environment).
exp = qx.Experiment(
    chip_id="64Qv3",
    muxes=[2],
    config_dir="/home/shared/qubex-config/64Qv3/config",
    params_dir="/home/shared/qubex-config/64Qv3/params",
)

In [ ]:
# Hardware connection is required for actual capture.
exp.connect()

In [ ]:
Q = exp.qubit_labels
R = exp.resonator_labels

schedule = qx.PulseSchedule()
with schedule as s:
    s.add(Q[0], qx.pulse.FlatTop(duration=112, amplitude=0.2, tau=0))
    s.add(Q[1], qx.pulse.Blank(duration=220))
    s.add(Q[1], qx.pulse.FlatTop(duration=128, amplitude=0.2, tau=0))
    s.add(Q[2], qx.pulse.Blank(duration=440))
    s.add(Q[2], qx.pulse.FlatTop(duration=144, amplitude=0.2, tau=0))
    s.add(Q[3], qx.pulse.Blank(duration=700))
    s.add(Q[3], qx.pulse.FlatTop(duration=160, amplitude=0.2, tau=0))

    s.add(R[0], qx.pulse.Blank(duration=120))
    s.add(R[0], qx.pulse.FlatTop(duration=320, amplitude=0.2, tau=0))
    s.add(R[1], qx.pulse.Blank(duration=180))
    s.add(R[1], qx.pulse.FlatTop(duration=320, amplitude=0.2, tau=0))
    s.add(R[2], qx.pulse.Blank(duration=620))
    s.add(R[2], qx.pulse.FlatTop(duration=280, amplitude=0.2, tau=0))
    s.add(R[3], qx.pulse.Blank(duration=980))
    s.add(R[3], qx.pulse.FlatTop(duration=240, amplitude=0.2, tau=0))

schedule.plot()

In [ ]:
# Full-span loopback capture over the whole schedule duration.
result = exp.capture_loopback(
    schedule=schedule,
    n_shots=4096,
)

print(f"sampling_period_ns = {result.sampling_period_ns}")
for target, captures in result.data.items():
    shapes = [np.asarray(cap).shape for cap in captures]
    print(target, shapes)

# Visualize each captured waveform.
result.plot()